# 05 — Model Comparison
Compare PhenoAge baseline formula against ML algorithms for biological age prediction.

**Target**: Minimize MAE (Mean Absolute Error) between predicted and chronological age.

**Algorithms**:
1. PhenoAge Formula (published baseline)
2. ElasticNet Regression
3. Ridge Regression
4. Random Forest
5. XGBoost
6. LightGBM

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import ElasticNet, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style='whitegrid', font_scale=1.1)
print('All libraries loaded.')

All libraries loaded.


## 1. Load clean data

In [2]:
df = pd.read_csv('../../data/processed/bioage_final_clean.csv')
print(f'Clean data: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Age range: {df.Age.min()} - {df.Age.max()}')
df.head()

Clean data: 19992 rows x 13 columns
Age range: 18.0 - 80.0


,Age,LBXSAL,LBXSCR,LBXGLU,CRP,LBXLYPCT,LBXMCVSI,LBXRDW,LBXSAPSI,LBXWBCSI,LBXGH,LBDHDD,LBXTC
0,44.0,3.5,0.8,90.0,2.44,35.8,80.1,13.7,74.0,5.3,6.0,39.0,105.0
1,70.0,5.0,1.2,157.0,0.05,29.4,90.3,12.5,48.0,7.5,7.1,59.0,147.0
2,73.0,3.9,1.2,100.0,0.21,29.1,88.6,13.4,77.0,6.6,5.9,49.0,186.0
3,79.0,4.1,0.9,97.0,0.20,18.8,97.2,12.4,67.0,6.5,5.0,81.0,181.0
4,59.0,3.7,0.9,86.0,0.15,37.8,85.9,13.6,99.0,4.3,5.8,76.0,205.0


## 2. Prepare features and target

In [3]:
# Features and target
target = 'Age'
feature_cols = [c for c in df.columns if c != target]

X = df[feature_cols].values
y = df[target].values

print(f'Features: {feature_cols}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

Features: ['LBXSAL', 'LBXSCR', 'LBXGLU', 'CRP', 'LBXLYPCT', 'LBXMCVSI', 'LBXRDW', 'LBXSAPSI', 'LBXWBCSI', 'LBXGH', 'LBDHDD', 'LBXTC']
X shape: (19992, 12)
y shape: (19992,)


## 3. PhenoAge Baseline Formula

The PhenoAge formula from Levine et al. (2018) is a **two-step process**.
Implementation adapted from [fangtiancheng/phenoage](https://github.com/fangtiancheng/phenoage).

**Step 1**: Compute log-odds of 10-year mortality (note: **Chronological Age is an input feature**).
```
xb = -19.907 - 0.0336*ALB(g/L) + 0.0095*CRE(µmol/L) + 0.1953*GLU(mmol/L)
     + 0.0954*ln(CRP/10) - 0.0120*LYMPH% + 0.0268*MCV
     + 0.3306*RDW + 0.0019*ALP + 0.0554*WBC + 0.0804*Age
```

**Step 2**: Convert to biological age via Gompertz CDF.
```
gamma = 0.0076927
M = 1 - exp(-exp(xb) * (exp(120*gamma) - 1) / gamma)
PhenoAge = 141.50225 + ln(-0.00553 * ln(1 - M)) / 0.090165
```

In [4]:
# PhenoAge formula from Levine et al. 2018 (correct implementation)
# Adapted from https://github.com/fangtiancheng/phenoage
#
# NOTE: NHANES units must be converted:
#   LBXSAL (g/dL) -> g/L:   multiply by 10
#   LBXSCR (mg/dL) -> µmol/L: multiply by 88.4
#   LBXGLU (mg/dL) -> mmol/L: divide by 18
#   CRP (mg/L) -> mg/dL: divide by 10, then take ln
#   LBXWBCSI (1000 cells/uL) = 10^9/L (same unit)

def phenoage_predict(row):
    """Apply the correct two-step PhenoAge formula."""
    # Unit conversions from NHANES to formula units
    albumin = row['LBXSAL'] * 10          # g/dL -> g/L
    creatinine = row['LBXSCR'] * 88.4     # mg/dL -> µmol/L
    glucose = row['LBXGLU'] / 18.0        # mg/dL -> mmol/L
    crp_log = np.log(row['CRP'] / 10.0)   # mg/L -> mg/dL -> ln
    lymph = row['LBXLYPCT']               # %
    mcv = row['LBXMCVSI']                 # fL
    rdw = row['LBXRDW']                   # %
    alp = row['LBXSAPSI']                 # U/L
    wbc = row['LBXWBCSI']                 # 10^9/L
    age = row['Age']                      # years (chronological age is an INPUT)

    # Step 1: Compute log-odds mortality score
    xb = (-19.907
           - 0.0336 * albumin
           + 0.0095 * creatinine
           + 0.1953 * glucose
           + 0.0954 * crp_log
           - 0.0120 * lymph
           + 0.0268 * mcv
           + 0.3306 * rdw
           + 0.0019 * alp
           + 0.0554 * wbc
           + 0.0804 * age)

    # Step 2: Convert to biological age via Gompertz CDF
    gamma = 0.0076927
    M = 1.0 - np.exp(-np.exp(xb) * (np.exp(120 * gamma) - 1) / gamma)
    M = np.clip(M, 1e-10, 1.0 - 1e-10)
    bioage = 141.50225 + np.log(-0.00553 * np.log(1.0 - M)) / 0.090165
    return bioage

# Apply to all rows
df['phenoage_pred'] = df.apply(phenoage_predict, axis=1)

# Evaluate
mae = mean_absolute_error(y, df['phenoage_pred'].values)
rmse = np.sqrt(mean_squared_error(y, df['phenoage_pred'].values))
r2 = r2_score(y, df['phenoage_pred'].values)

print(f'PhenoAge Formula Results:')
print(f'  MAE:  {mae:.3f} years')
print(f'  RMSE: {rmse:.3f} years')
print(f'  R²:   {r2:.4f}')

PhenoAge Formula Results:
  MAE:  7.186 years
  RMSE: 9.245 years
  R²:   0.7507


## 4. Cross-Validation Setup

In [5]:
N_SPLITS = 5
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
print(f'{N_SPLITS}-fold cross-validation')
print(f'Train size per fold: ~{int(len(X) * 0.8 / N_SPLITS)} samples')

5-fold cross-validation
Train size per fold: ~3198 samples


## 5. Algorithm Comparison

In [6]:
# Define models
models = {
    'ElasticNet': Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000))
    ]),
    'Ridge': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0))
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_leaf=10,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    ),
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1
    ),
}

print(f'Defined {len(models)} models for comparison.')

Defined 5 models for comparison.


In [7]:
# Cross-validate each model
results = []

# Add PhenoAge baseline
pheno_mae = mean_absolute_error(y, df['phenoage_pred'].values)
results.append({
    'Model': 'PhenoAge Formula',
    'MAE_mean': pheno_mae,
    'MAE_std': 0,
    'RMSE_mean': np.sqrt(mean_squared_error(y, df['phenoage_pred'].values)),
    'R2_mean': r2_score(y, df['phenoage_pred'].values),
})

for name, model in models.items():
    print(f'Training {name}...')
    
    # MAE scores
    mae_scores = -cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    
    # RMSE scores
    rmse_scores = np.sqrt(-cross_val_score(model, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1))
    
    # R2 scores
    r2_scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
    
    results.append({
        'Model': name,
        'MAE_mean': mae_scores.mean(),
        'MAE_std': mae_scores.std(),
        'RMSE_mean': rmse_scores.mean(),
        'R2_mean': r2_scores.mean(),
    })
    print(f'  MAE: {mae_scores.mean():.3f} +/- {mae_scores.std():.3f}')

results_df = pd.DataFrame(results).sort_values('MAE_mean')
print('\n=== Results Summary ===')
print(results_df.to_string(index=False))

Training ElasticNet...
  MAE: 12.986 +/- 0.225
Training Ridge...
  MAE: 12.873 +/- 0.222
Training Random Forest...
  MAE: 9.883 +/- 0.154
Training XGBoost...
  MAE: 9.550 +/- 0.079
Training LightGBM...
  MAE: 9.661 +/- 0.117

=== Results Summary ===
           Model  MAE_mean  MAE_std  RMSE_mean  R2_mean
PhenoAge Formula  7.185616 0.000000   9.244655 0.750679
         XGBoost  9.550139 0.079116  12.179856 0.567069
        LightGBM  9.660969 0.116812  12.273760 0.560389
   Random Forest  9.883260 0.154131  12.458212 0.547072
           Ridge 12.872982 0.222259  15.550067 0.294320
      ElasticNet 12.986125 0.225355  15.576669 0.291925


## 6. Results Visualization

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# MAE comparison
ax = axes[0]
colors = ['#2ecc71' if m == results_df.Model.iloc[0] else '#3498db' for m in results_df.Model]
bars = ax.barh(results_df.Model, results_df.MAE_mean, xerr=results_df.MAE_std, color=colors)
ax.set_xlabel('MAE (years)')
ax.set_title('Mean Absolute Error (lower = better)')
ax.invert_yaxis()
for bar, val in zip(bars, results_df.MAE_mean):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')

# RMSE comparison
ax = axes[1]
bars = ax.barh(results_df.Model, results_df.RMSE_mean, color=colors)
ax.set_xlabel('RMSE (years)')
ax.set_title('Root Mean Squared Error (lower = better)')
ax.invert_yaxis()
for bar, val in zip(bars, results_df.RMSE_mean):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')

# R2 comparison
ax = axes[2]
bars = ax.barh(results_df.Model, results_df.R2_mean, color=colors)
ax.set_xlabel('R² Score')
ax.set_title('R² Score (higher = better)')
ax.invert_yaxis()
for bar, val in zip(bars, results_df.R2_mean):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')

plt.tight_layout()
plt.savefig('../../reports/figures/14_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/14_model_comparison.png')

Saved: reports/figures/14_model_comparison.png


## 7. Best Model — Detailed Analysis

In [9]:
# Get best model
best_row = results_df.iloc[0]
best_name = best_row['Model']
print(f'Best model: {best_name}')
print(f'  MAE:  {best_row.MAE_mean:.3f} years')
print(f'  RMSE: {best_row.RMSE_mean:.3f} years')
print(f'  R²:   {best_row.R2_mean:.4f}')

Best model: PhenoAge Formula
  MAE:  7.186 years
  RMSE: 9.245 years
  R²:   0.7507


In [10]:
# Train best model on full data and show predictions
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['XGBoost']

# Fit on full data for visualization
best_model.fit(X, y)
y_pred = best_model.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
ax = axes[0]
ax.scatter(y, y_pred, alpha=0.2, s=10)
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Chronological Age')
ax.set_ylabel('Predicted Biological Age')
ax.set_title(f'{best_name}: Predicted vs Actual')
ax.legend()

# BioAge Gap distribution
ax = axes[1]
bioage_gap = y_pred - y
ax.hist(bioage_gap, bins=50, edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('BioAge Gap (Predicted - Chronological)')
ax.set_ylabel('Count')
ax.set_title('BioAge Gap Distribution')
ax.text(0.02, 0.95, f'Mean gap: {bioage_gap.mean():.2f} years\nStd: {bioage_gap.std():.2f} years',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('../../reports/figures/15_best_model_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/15_best_model_analysis.png')

Saved: reports/figures/15_best_model_analysis.png


## 8. Feature Importance (Tree-based models)

In [11]:
# Feature importance from best tree-based model
if best_name in ['Random Forest', 'XGBoost', 'LightGBM']:
    importances = best_model.feature_importances_
else:
    # Use XGBoost for feature importance
    xgb_model = models['XGBoost']
    xgb_model.fit(X, y)
    importances = xgb_model.feature_importances_

feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_imp.Feature, feat_imp.Importance, color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance (XGBoost)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../../reports/figures/16_feature_importance.png', dpi=150)
plt.show()
print('Saved: reports/figures/16_feature_importance.png')

Saved: reports/figures/16_feature_importance.png


## 9. Final Summary

In [12]:
print('='*60)
print('VITALAGE MODEL COMPARISON SUMMARY')
print('='*60)
print()
print('Dataset: NHANES 2005-2023, Ages 18-80')
print(f'Samples: {len(df)} | Features: {len(feature_cols)}')
print()
print('Results (5-fold CV):')
print(results_df.to_string(index=False))
print()
print(f'Best Model: {best_name}')
print(f'  MAE = {best_row.MAE_mean:.3f} +/- {best_row.MAE_std:.3f} years')
print(f'  This means predictions are off by ~{best_row.MAE_mean:.1f} years on average.')
print()
if best_row.MAE_mean < 5:
    print('VERDICT: Good model performance for a hackathon.')
elif best_row.MAE_mean < 7:
    print('VERDICT: Acceptable. Consider feature engineering or hyperparameter tuning.')
else:
    print('VERDICT: Needs improvement. Try adding more features or imputation.')

VITALAGE MODEL COMPARISON SUMMARY

Dataset: NHANES 2005-2023, Ages 18-80
Samples: 19992 | Features: 12

Results (5-fold CV):
           Model  MAE_mean  MAE_std  RMSE_mean  R2_mean
PhenoAge Formula  7.185616 0.000000   9.244655 0.750679
         XGBoost  9.550139 0.079116  12.179856 0.567069
        LightGBM  9.660969 0.116812  12.273760 0.560389
   Random Forest  9.883260 0.154131  12.458212 0.547072
           Ridge 12.872982 0.222259  15.550067 0.294320
      ElasticNet 12.986125 0.225355  15.576669 0.291925

Best Model: PhenoAge Formula
  MAE = 7.186 +/- 0.000 years
  This means predictions are off by ~7.2 years on average.

VERDICT: Needs improvement. Try adding more features or imputation.


## Next Steps
1. Tune hyperparameters of the best model (GridSearchCV)
2. Add interaction features (e.g., CRP × Albumin)
3. Try imputation instead of complete-case to keep more rows
4. Export the trained model as .pkl for the Streamlit app
5. Compute BioAge Gap for each participant and analyze risk groups